# Prithvi-EO-2.0 草原火灾 Patch 分类训练与预测分析正式版

这个 notebook 基于已经跑通的 `prithvi_eo_v2_300` backbone 和现有分类头权重 `prithvi_patch_classifier_head.pth` 整理而成。

流程分成四块：

1. 匹配 6 波段影像与火灾 label，按 label 生成 patch 样本。
2. 冻结 Prithvi backbone，只训练 patch 分类头。
3. 在验证集保存最优分类头，并在测试集做指标分析。
4. 对整幅影像做滑窗预测，输出火灾概率图和预测结果表。

第一次运行时，先检查“配置区”的路径和类别名称。


In [ ]:
# =========================
# 0. 环境检查与基础导入
# =========================

import os
import re
import random
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

print("Python:", os.sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
print("CUDA 版本:", torch.version.cuda)
print("terratorch:", terratorch.__version__ if hasattr(terratorch, "__version__") else "ok")


In [ ]:
# =========================
# 1. 配置区：主要改这里
# =========================

# 数据路径。代码会优先使用存在的路径；如果你换了数据目录，只需要改这里。
IMAGE_DIR_CANDIDATES = [
    Path(r"E:\satellite_data\02grassland_fire\image_6band_without_smoke"),
    Path(r"E:\satellite_data\02grassland_fire\image_6band"),
]

LABEL_DIR_CANDIDATES = [
    Path(r"E:\satellite_data\02grassland_fire\label_6band_without_smoke"),
    Path(r"E:\satellite_data\02grassland_fire\label_6band"),
]

# Prithvi-EO-2.0 300M，和你已经跑通的 notebook 保持一致。
MODEL_NAME = "prithvi_eo_v2_300"
PRETRAINED = True
FREEZE_BACKBONE = True
# Keep most of Prithvi frozen, but fine-tune the last transformer block.
UNFREEZE_LAST_BACKBONE_BLOCKS = 2
BACKBONE_LEARNING_RATE = 5e-6

# 已有分类头，可用于继续训练或直接预测。
INIT_HEAD_PATH = Path("prithvi_patch_classifier_head.pth")
LOAD_EXISTING_HEAD = INIT_HEAD_PATH.exists()

# 新训练出来的最优分类头会保存在这里。
BEST_HEAD_PATH = Path("prithvi_patch_classifier_head_formal_best.pth")
PREDICTION_CSV_PATH = Path("prithvi_patch_predictions.csv")
SCENE_PROB_TIF_PATH = Path("prithvi_scene_fire_probability.tif")

# 类别顺序必须和分类头输出顺序一致：0=负样本，1=正样本。
CLASS_NAMES = ["no_fire", "fire"]
NUM_CLASSES = len(CLASS_NAMES)
POSITIVE_CLASS_INDEX = 1

# ===== Recall-first fire warning settings =====
# Lower threshold means fewer missed fires and more false alarms.
FIRE_THRESHOLD = 0.1
FBETA_BETA = 2.0
POSITIVE_WEIGHT_MULTIPLIER = 4.0

# 文件名匹配：lead_days=3 表示用影像日期之后至少 3 天的 label 做监督。
PAIR_MODE = "lead_days"     # 可选："lead_days" 或 "same_stem"
LEAD_DAYS = 3

# Patch 参数。Prithvi 这版已经验证过 224x224 输入，建议先保持 224。
PATCH_SIZE = 224
STRIDE = 112
MODEL_INPUT_SIZE = 224
BANDS = [1, 2, 3, 4, 5, 6]  # rasterio 是 1-based；顺序：BLUE/GREEN/RED/NIR/SWIR1/SWIR2

# 样本生成策略。
MIN_POSITIVE_PIXELS = 3
NEGATIVE_KEEP_RATIO = 0.15
MAX_PATCHES_PER_IMAGE = 600

# 训练参数。
RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
BATCH_SIZE = 1
PRED_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # Windows + notebook 建议先用 0

# 影像预处理。HLS 反射率常见是 0-10000，先缩放到 0-1。
NORMALIZE_MODE = "reflectance_10000"  # 可选："reflectance_10000", "percentile", "none"
DISPLAY_BANDS_ZERO_BASED = [3, 2, 1]  # NIR/RED/GREEN false color；真彩色可改成 [2, 1, 0]


def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


IMAGE_DIR = first_existing_path(IMAGE_DIR_CANDIDATES)
LABEL_DIR = first_existing_path(LABEL_DIR_CANDIDATES)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DEVICE_TYPE = "cuda" if DEVICE.type == "cuda" else "cpu"

print("DEVICE:", DEVICE)
print("IMAGE_DIR:", IMAGE_DIR, "exists=", IMAGE_DIR.exists())
print("LABEL_DIR:", LABEL_DIR, "exists=", LABEL_DIR.exists())
print("LOAD_EXISTING_HEAD:", LOAD_EXISTING_HEAD, INIT_HEAD_PATH)
print("FIRE_THRESHOLD:", FIRE_THRESHOLD, "FBETA_BETA:", FBETA_BETA)
print("POSITIVE_WEIGHT_MULTIPLIER:", POSITIVE_WEIGHT_MULTIPLIER)
print("UNFREEZE_LAST_BACKBONE_BLOCKS:", UNFREEZE_LAST_BACKBONE_BLOCKS)
print("BATCH_SIZE:", BATCH_SIZE, "PRED_BATCH_SIZE:", PRED_BATCH_SIZE, "GRAD_ACCUM:", GRADIENT_ACCUMULATION_STEPS)


In [ ]:
# =========================
# 2. 随机种子与影像/label 匹配
# =========================

TIME_PATTERN = re.compile(r"(\d{8}T\d{6})")
SPATIAL_KEY_PATTERN = re.compile(r"_(T\d{2}[A-Z]{3})_\d{8}T\d{6}_(R\d+_C\d+_\d+)$")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)


def extract_acquisition_time(path: Path):
    matches = TIME_PATTERN.findall(path.stem)
    if not matches:
        return None
    return datetime.strptime(matches[0], "%Y%m%dT%H%M%S")


def extract_spatial_key(path: Path):
    match = SPATIAL_KEY_PATTERN.search(path.stem)
    if match is None:
        return None
    return f"{match.group(1)}_{match.group(2)}"


def match_same_stem(image_dir: Path, label_dir: Path):
    image_files = sorted(image_dir.glob("*.tif")) + sorted(image_dir.glob("*.tiff"))
    label_files = sorted(label_dir.glob("*.tif")) + sorted(label_dir.glob("*.tiff"))
    label_by_stem = {p.stem: p for p in label_files}
    return [(img, label_by_stem[img.stem]) for img in image_files if img.stem in label_by_stem]


def match_by_lead_days(image_dir: Path, label_dir: Path, lead_days: int):
    image_files = sorted(image_dir.glob("*.tif")) + sorted(image_dir.glob("*.tiff"))
    label_files = sorted(label_dir.glob("*.tif")) + sorted(label_dir.glob("*.tiff"))

    label_groups = {}
    skipped_label_files = 0
    for lab_path in label_files:
        lab_time = extract_acquisition_time(lab_path)
        spatial_key = extract_spatial_key(lab_path)
        if lab_time is None or spatial_key is None:
            skipped_label_files += 1
            continue
        label_groups.setdefault(spatial_key, []).append((lab_time, lab_path))

    for spatial_key in label_groups:
        label_groups[spatial_key].sort(key=lambda item: item[0])

    pairs = []
    lead_offsets = []
    skipped_image_files = 0
    for img_path in image_files:
        img_time = extract_acquisition_time(img_path)
        spatial_key = extract_spatial_key(img_path)
        if img_time is None or spatial_key is None:
            skipped_image_files += 1
            continue

        target_time = img_time + timedelta(days=lead_days)
        candidates = label_groups.get(spatial_key, [])
        future_candidates = [(lab_time, lab_path) for lab_time, lab_path in candidates if lab_time >= target_time]
        if future_candidates:
            matched_time, matched_label_path = min(future_candidates, key=lambda item: item[0])
            pairs.append((img_path, matched_label_path))
            lead_offsets.append((matched_time - img_time).days)

    print("skipped_image_files:", skipped_image_files)
    print("skipped_label_files:", skipped_label_files)
    if lead_offsets:
        print("actual lead days min/max:", min(lead_offsets), max(lead_offsets))
        print("actual lead days examples:", sorted(set(lead_offsets))[:10])
    return pairs


def build_pairs():
    if not IMAGE_DIR.exists() or not LABEL_DIR.exists():
        raise FileNotFoundError("IMAGE_DIR 或 LABEL_DIR 不存在，请先在配置区改路径。")

    if PAIR_MODE == "same_stem":
        pairs = match_same_stem(IMAGE_DIR, LABEL_DIR)
    elif PAIR_MODE == "lead_days":
        pairs = match_by_lead_days(IMAGE_DIR, LABEL_DIR, LEAD_DAYS)
        if len(pairs) == 0:
            print("lead_days 未匹配到样本，尝试 same_stem 匹配。")
            pairs = match_same_stem(IMAGE_DIR, LABEL_DIR)
    else:
        raise ValueError(f"未知 PAIR_MODE: {PAIR_MODE}")

    if len(pairs) == 0:
        raise ValueError("没有匹配到 image-label pairs，请检查路径、文件名和 PAIR_MODE。")

    random.shuffle(pairs)
    return pairs


def split_pairs(pairs):
    n_total = len(pairs)
    n_train = int(n_total * TRAIN_RATIO)
    n_val = int(n_total * VAL_RATIO)

    if n_total >= 3:
        n_train = max(1, n_train)
        n_val = max(1, n_val)
        if n_train + n_val >= n_total:
            n_val = max(1, n_total - n_train - 1)
    else:
        n_train = max(1, n_total - 1)
        n_val = 0

    train_pairs = pairs[:n_train]
    val_pairs = pairs[n_train:n_train + n_val]
    test_pairs = pairs[n_train + n_val:]
    return train_pairs, val_pairs, test_pairs


pairs = build_pairs()
train_pairs, val_pairs, test_pairs = split_pairs(pairs)

print("matched pairs:", len(pairs))
print("train_pairs:", len(train_pairs))
print("val_pairs:", len(val_pairs))
print("test_pairs:", len(test_pairs))
print("example pair:", pairs[0][0].name, "->", pairs[0][1].name)


In [ ]:
# =========================
# 3. 由 label 生成 patch 样本
# =========================


def build_samples(pair_list, set_name="train"):
    samples = []

    for img_path, lab_path in tqdm(pair_list, desc=f"build {set_name} samples"):
        with rasterio.open(lab_path) as lab_src:
            label = lab_src.read(1)

        height, width = label.shape
        coords = []

        for y in range(0, height - PATCH_SIZE + 1, STRIDE):
            for x in range(0, width - PATCH_SIZE + 1, STRIDE):
                lab_patch = label[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                positive_pixels = int(np.sum(lab_patch > 0))

                if positive_pixels >= MIN_POSITIVE_PIXELS:
                    cls = POSITIVE_CLASS_INDEX
                    coords.append((img_path, lab_path, x, y, cls, positive_pixels))
                elif random.random() < NEGATIVE_KEEP_RATIO:
                    cls = 0
                    coords.append((img_path, lab_path, x, y, cls, positive_pixels))

        if MAX_PATCHES_PER_IMAGE is not None and len(coords) > MAX_PATCHES_PER_IMAGE:
            positive_coords = [item for item in coords if item[4] == POSITIVE_CLASS_INDEX]
            negative_coords = [item for item in coords if item[4] != POSITIVE_CLASS_INDEX]
            negative_quota = max(MAX_PATCHES_PER_IMAGE - len(positive_coords), 0)
            if len(negative_coords) > negative_quota:
                negative_coords = random.sample(negative_coords, negative_quota)
            coords = positive_coords + negative_coords

        samples.extend(coords)

    labels = [s[4] for s in samples]
    positive_num = sum(label == POSITIVE_CLASS_INDEX for label in labels)
    negative_num = sum(label == 0 for label in labels)

    print(f"{set_name} patch 总数:", len(samples))
    print(f"{set_name} 正样本数:", positive_num)
    print(f"{set_name} 负样本数:", negative_num)
    return samples


train_samples = build_samples(train_pairs, "训练集")
val_samples = build_samples(val_pairs, "验证集") if len(val_pairs) > 0 else []
test_samples = build_samples(test_pairs, "测试集") if len(test_pairs) > 0 else []

if len(train_samples) == 0:
    raise ValueError("训练集没有生成任何 patch，请检查 label 或调小 PATCH_SIZE / MIN_POSITIVE_PIXELS。")

if len(val_samples) == 0:
    print("警告：验证集 patch 为空，训练时会无法选择 best head。建议增加数据或调整 split。")


In [ ]:
# =========================
# 4. Dataset / DataLoader
# =========================


def preprocess_patch(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

    if NORMALIZE_MODE == "reflectance_10000":
        img = np.clip(img, 0, 10000) / 10000.0
    elif NORMALIZE_MODE == "percentile":
        scaled = np.zeros_like(img, dtype=np.float32)
        for band_idx in range(img.shape[0]):
            band = img[band_idx]
            valid = band[np.isfinite(band)]
            if valid.size == 0:
                continue
            low, high = np.percentile(valid, [2, 98])
            if high <= low:
                scaled[band_idx] = 0
            else:
                scaled[band_idx] = np.clip((band - low) / (high - low), 0, 1)
        img = scaled
    elif NORMALIZE_MODE == "none":
        pass
    else:
        raise ValueError(f"未知 NORMALIZE_MODE: {NORMALIZE_MODE}")

    return img


def resize_for_prithvi(x_tensor: torch.Tensor) -> torch.Tensor:
    if x_tensor.shape[-2:] == (MODEL_INPUT_SIZE, MODEL_INPUT_SIZE):
        return x_tensor
    x_tensor = F.interpolate(
        x_tensor.unsqueeze(0),
        size=(MODEL_INPUT_SIZE, MODEL_INPUT_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)
    return x_tensor


class FirePatchDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lab_path, x, y, cls, positive_pixels = self.samples[idx]

        with rasterio.open(img_path) as src:
            if src.count < max(BANDS):
                raise ValueError(f"{img_path} 只有 {src.count} 个波段，但 BANDS={BANDS}")
            img = src.read(indexes=BANDS, window=Window(x, y, PATCH_SIZE, PATCH_SIZE))

        img = preprocess_patch(img)
        image_tensor = torch.tensor(img, dtype=torch.float32)
        image_tensor = resize_for_prithvi(image_tensor)

        return {
            "image": image_tensor,
            "label": torch.tensor(cls, dtype=torch.long),
            "image_path": str(img_path),
            "label_path": str(lab_path),
            "x": int(x),
            "y": int(y),
            "positive_pixels": int(positive_pixels),
        }


def make_loader(samples, shuffle=False, batch_size=None):
    dataset = FirePatchDataset(samples)
    return DataLoader(
        dataset,
        batch_size=batch_size or BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )


train_loader = make_loader(train_samples, shuffle=True, batch_size=BATCH_SIZE)
val_loader = make_loader(val_samples, shuffle=False, batch_size=BATCH_SIZE) if len(val_samples) else None
test_loader = make_loader(test_samples, shuffle=False, batch_size=BATCH_SIZE) if len(test_samples) else None

batch = next(iter(train_loader))
print("batch image:", batch["image"].shape)
print("batch label:", batch["label"].shape)


In [ ]:
# =========================
# 5. Prithvi backbone + ???????
# =========================


def extract_feature_from_output(out):
    if isinstance(out, torch.Tensor):
        feat = out
    elif isinstance(out, (list, tuple)):
        tensor_items = [item for item in out if isinstance(item, torch.Tensor)]
        if len(tensor_items) == 0:
            raise ValueError("Prithvi ?? list/tuple ??? tensor")
        feat = tensor_items[-1]
    elif isinstance(out, dict):
        possible_keys = ["last_hidden_state", "features", "out", "embedding", "x"]
        feat = None
        for key in possible_keys:
            if key in out and isinstance(out[key], torch.Tensor):
                feat = out[key]
                break
        if feat is None:
            for value in out.values():
                if isinstance(value, torch.Tensor):
                    feat = value
                    break
        if feat is None:
            raise ValueError("Prithvi ?? dict ??? tensor")
    else:
        raise TypeError(f"?????????: {type(out)}")
    return feat


def pool_feature(feat):
    if feat.ndim == 2:
        return feat
    if feat.ndim == 3:
        return feat.mean(dim=1)
    if feat.ndim == 4:
        return feat.mean(dim=(2, 3))
    if feat.ndim == 5:
        return feat.mean(dim=(1, 3, 4))
    raise ValueError(f"?????????: {feat.shape}")


def set_module_trainable(module, requires_grad=True):
    for param in module.parameters():
        param.requires_grad = requires_grad


def resolve_module_path(root, path):
    module = root
    for name in path.split("."):
        if not hasattr(module, name):
            return None
        module = getattr(module, name)
    return module


def find_transformer_blocks(backbone):
    candidate_paths = [
        "blocks",
        "encoder.blocks",
        "encoder.layers",
        "model.blocks",
        "model.encoder.layers",
        "backbone.blocks",
    ]

    for path in candidate_paths:
        module = resolve_module_path(backbone, path)
        if isinstance(module, (nn.ModuleList, nn.Sequential, list, tuple)) and len(module) > 0:
            if sum(p.numel() for p in module[-1].parameters()) > 0:
                return path, module

    for name, module in backbone.named_modules():
        if isinstance(module, (nn.ModuleList, nn.Sequential)) and len(module) > 0:
            if sum(p.numel() for p in module[-1].parameters()) > 0:
                return name, module

    return None, None


def configure_backbone_trainability(backbone, freeze_backbone=True, unfreeze_last_n_blocks=0):
    trainable_modules = []
    trainable_names = []

    if not freeze_backbone:
        set_module_trainable(backbone, True)
        return [backbone], ["full_backbone"]

    set_module_trainable(backbone, False)

    if unfreeze_last_n_blocks <= 0:
        backbone.eval()
        return trainable_modules, trainable_names

    blocks_path, blocks = find_transformer_blocks(backbone)
    if blocks is None:
        print("WARNING: Could not find transformer blocks. Backbone remains frozen.")
        backbone.eval()
        return trainable_modules, trainable_names

    n_blocks = min(unfreeze_last_n_blocks, len(blocks))
    for idx in range(len(blocks) - n_blocks, len(blocks)):
        block = blocks[idx]
        set_module_trainable(block, True)
        trainable_modules.append(block)
        trainable_names.append(f"{blocks_path}[{idx}]")

    # Final norms are tiny and usually helpful to tune with the last block.
    for norm_name in ["norm", "fc_norm"]:
        norm_module = getattr(backbone, norm_name, None)
        if norm_module is not None:
            set_module_trainable(norm_module, True)
            trainable_modules.append(norm_module)
            trainable_names.append(norm_name)

    backbone.eval()
    for module in trainable_modules:
        module.train()

    return trainable_modules, trainable_names


class PatchClassifierHead(nn.Module):
    def __init__(self, feature_dim=1024, hidden_dim=128, num_classes=2, dropout=0.30):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.classifier(x)


class PrithviPatchClassifier(nn.Module):
    def __init__(
        self,
        model_name="prithvi_eo_v2_300",
        pretrained=True,
        feature_dim=1024,
        hidden_dim=128,
        num_classes=2,
        dropout=0.30,
        freeze_backbone=True,
        unfreeze_last_n_blocks=0,
    ):
        super().__init__()
        self.freeze_backbone = freeze_backbone
        self.unfreeze_last_n_blocks = unfreeze_last_n_blocks
        self.backbone = BACKBONE_REGISTRY.build(model_name, pretrained=pretrained)
        self.head = PatchClassifierHead(feature_dim, hidden_dim, num_classes, dropout)

        self.trainable_backbone_modules, self.trainable_backbone_names = configure_backbone_trainability(
            self.backbone,
            freeze_backbone=freeze_backbone,
            unfreeze_last_n_blocks=unfreeze_last_n_blocks,
        )
        self.backbone_has_trainable_params = any(p.requires_grad for p in self.backbone.parameters())

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_backbone:
            self.backbone.eval()
            for module in self.trainable_backbone_modules:
                module.train(mode)
        return self

    def forward(self, x):
        if self.backbone_has_trainable_params:
            out = self.backbone(x)
        else:
            with torch.no_grad():
                out = self.backbone(x)

        feat = extract_feature_from_output(out)
        pooled = pool_feature(feat)
        logits = self.head(pooled)
        return logits


def infer_head_config_from_checkpoint(path: Path):
    if not path.exists():
        return 1024, 128, NUM_CLASSES

    ckpt = torch.load(path, map_location="cpu")
    state_dict = ckpt.get("classifier_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    first_weight = state_dict.get("classifier.0.weight")
    last_weight = state_dict.get("classifier.3.weight")

    feature_dim = int(ckpt.get("feature_dim", 1024)) if isinstance(ckpt, dict) else 1024
    hidden_dim = int(first_weight.shape[0]) if first_weight is not None else 128
    num_classes = int(last_weight.shape[0]) if last_weight is not None else NUM_CLASSES
    return feature_dim, hidden_dim, num_classes


FEATURE_DIM, HIDDEN_DIM, HEAD_NUM_CLASSES = infer_head_config_from_checkpoint(INIT_HEAD_PATH)
print("head config:", FEATURE_DIM, HIDDEN_DIM, HEAD_NUM_CLASSES)

model = PrithviPatchClassifier(
    model_name=MODEL_NAME,
    pretrained=PRETRAINED,
    feature_dim=FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=HEAD_NUM_CLASSES,
    dropout=0.30,
    freeze_backbone=FREEZE_BACKBONE,
    unfreeze_last_n_blocks=UNFREEZE_LAST_BACKBONE_BLOCKS,
).to(DEVICE)

print("PrithviPatchClassifier ready")
print("unfrozen backbone modules:", model.trainable_backbone_names)
print("backbone trainable params:", sum(p.numel() for p in model.backbone.parameters() if p.requires_grad))
print("head trainable params:", sum(p.numel() for p in model.head.parameters() if p.requires_grad))


In [ ]:
# =========================
# 6. 加载现有分类头权重，可跳过
# =========================


def load_classifier_head(model, checkpoint_path: Path, strict=True):
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    if isinstance(ckpt, dict) and "classifier_state_dict" in ckpt:
        state_dict = ckpt["classifier_state_dict"]
    elif isinstance(ckpt, dict) and "head_state_dict" in ckpt:
        state_dict = ckpt["head_state_dict"]
    else:
        state_dict = ckpt

    missing, unexpected = model.head.load_state_dict(state_dict, strict=strict)
    print("分类头加载完成:", checkpoint_path)
    if missing:
        print("missing keys:", missing)
    if unexpected:
        print("unexpected keys:", unexpected)
    return ckpt


if LOAD_EXISTING_HEAD:
    init_ckpt = load_classifier_head(model, INIT_HEAD_PATH, strict=True)
else:
    print("未加载现有分类头，将从随机初始化开始训练。")

# 快速前向检查
model.eval()
with torch.no_grad():
    logits = model(batch["image"][:1].to(DEVICE))
print("logits shape:", logits.shape)
print("prob:", torch.softmax(logits, dim=1).detach().cpu().numpy())


In [ ]:
# =========================
# 7. Loss, optimizer, and evaluation helpers
# =========================


def threshold_predict_from_probs(probs, threshold=None):
    """Use threshold alerts for binary classification; fall back to argmax for multiclass."""
    threshold = FIRE_THRESHOLD if threshold is None else threshold
    if probs.ndim == 2 and probs.shape[1] == 2:
        return (probs[:, POSITIVE_CLASS_INDEX] >= threshold).astype(np.int64)
    return probs.argmax(axis=1).astype(np.int64)


def threshold_predict_from_tensor(probs, threshold=None):
    threshold = FIRE_THRESHOLD if threshold is None else threshold
    if probs.ndim == 2 and probs.shape[1] == 2:
        return (probs[:, POSITIVE_CLASS_INDEX] >= threshold).long()
    return torch.argmax(probs, dim=1)


def binary_confidence_from_probs(probs, preds):
    if probs.ndim == 2 and probs.shape[1] == 2:
        fire_probs = probs[:, POSITIVE_CLASS_INDEX]
        return np.where(preds == POSITIVE_CLASS_INDEX, fire_probs, 1.0 - fire_probs)
    return probs.max(axis=1)


def make_class_weights(samples, num_classes):
    labels = np.array([s[4] for s in samples], dtype=np.int64)
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    total = counts.sum()
    weights = total / np.maximum(num_classes * counts, 1.0)
    if num_classes > POSITIVE_CLASS_INDEX:
        weights[POSITIVE_CLASS_INDEX] *= POSITIVE_WEIGHT_MULTIPLIER
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE), counts


class_weights, class_counts = make_class_weights(train_samples, HEAD_NUM_CLASSES)
print("train class counts:", class_counts)
print("class weights after positive multiplier:", class_weights.detach().cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=class_weights)
head_params = [p for p in model.head.parameters() if p.requires_grad]
backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
param_groups = [{"params": head_params, "lr": LEARNING_RATE}]
if backbone_params:
    param_groups.append({"params": backbone_params, "lr": BACKBONE_LEARNING_RATE})

optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DEVICE_TYPE == "cuda"))

print("optimizer head params:", sum(p.numel() for p in head_params))
print("optimizer backbone params:", sum(p.numel() for p in backbone_params))
print("optimizer learning rates:", [group["lr"] for group in optimizer.param_groups])


def compute_metrics(y_true, y_pred, y_prob=None, loss=np.nan):
    metrics = {
        "loss": loss,
        "acc": accuracy_score(y_true, y_pred) if len(y_true) else np.nan,
        "precision": precision_score(y_true, y_pred, pos_label=POSITIVE_CLASS_INDEX, zero_division=0) if len(y_true) else np.nan,
        "recall": recall_score(y_true, y_pred, pos_label=POSITIVE_CLASS_INDEX, zero_division=0) if len(y_true) else np.nan,
        "f1": f1_score(y_true, y_pred, pos_label=POSITIVE_CLASS_INDEX, zero_division=0) if len(y_true) else np.nan,
        "f2": fbeta_score(y_true, y_pred, beta=FBETA_BETA, pos_label=POSITIVE_CLASS_INDEX, zero_division=0) if len(y_true) else np.nan,
        "auc": np.nan,
    }
    if y_prob is not None and y_prob.ndim == 2 and y_prob.shape[1] > POSITIVE_CLASS_INDEX and len(np.unique(y_true)) == 2:
        metrics["auc"] = roc_auc_score(y_true, y_prob[:, POSITIVE_CLASS_INDEX])
    return metrics


def evaluate(model, loader, criterion=None, desc="eval"):
    if loader is None:
        return {"loss": np.nan, "acc": np.nan, "precision": np.nan, "recall": np.nan, "f1": np.nan, "f2": np.nan, "auc": np.nan}

    model.eval()
    total_loss = 0.0
    total_num = 0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, leave=False):
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast(AMP_DEVICE_TYPE, enabled=(AMP_DEVICE_TYPE == "cuda")):
                logits = model(images)
                loss = criterion(logits, labels) if criterion is not None else None
            probs = torch.softmax(logits, dim=1)
            preds = threshold_predict_from_tensor(probs)
            if loss is not None:
                total_loss += loss.item() * images.size(0)
            total_num += images.size(0)
            all_labels.extend(labels.detach().cpu().numpy().tolist())
            all_preds.extend(preds.detach().cpu().numpy().tolist())
            all_probs.extend(probs.detach().cpu().numpy().tolist())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)
    avg_loss = total_loss / max(total_num, 1) if criterion is not None else np.nan
    return compute_metrics(y_true, y_pred, y_prob, loss=avg_loss)


def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    total_loss = 0.0
    total_num = 0
    all_labels = []
    all_preds = []

    optimizer.zero_grad(set_to_none=True)
    num_batches = len(loader)

    for step, batch in enumerate(tqdm(loader, desc=f"train epoch {epoch}", leave=False), start=1):
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(AMP_DEVICE_TYPE, enabled=(AMP_DEVICE_TYPE == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)
            backward_loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(backward_loss).backward()

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == num_batches:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        probs = torch.softmax(logits.detach(), dim=1)
        preds = threshold_predict_from_tensor(probs)
        total_loss += loss.item() * images.size(0)
        total_num += images.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    avg_loss = total_loss / max(total_num, 1)
    return compute_metrics(y_true, y_pred, loss=avg_loss)


In [ ]:
# =========================
# 8. Formal training: save the best validation F2 classifier head
# =========================

history = []
best_val_f2 = -1.0

if val_loader is None:
    raise ValueError("val_loader is empty. Adjust TRAIN_RATIO / VAL_RATIO or reduce PATCH_SIZE.")

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch)
    val_metrics = evaluate(model, val_loader, criterion=criterion, desc=f"val epoch {epoch}")
    scheduler.step(val_metrics["f2"])

    row = {
        "epoch": epoch,
        "head_lr": optimizer.param_groups[0]["lr"],
        "backbone_lr": optimizer.param_groups[1]["lr"] if len(optimizer.param_groups) > 1 else 0.0,
        "fire_threshold": FIRE_THRESHOLD,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)

    print(
        f"Epoch [{epoch}/{NUM_EPOCHS}] "
        f"Train Loss={train_metrics['loss']:.4f} Acc={train_metrics['acc']:.4f} "
        f"Recall={train_metrics['recall']:.4f} F2={train_metrics['f2']:.4f} | "
        f"Val Loss={val_metrics['loss']:.4f} Acc={val_metrics['acc']:.4f} "
        f"P={val_metrics['precision']:.4f} R={val_metrics['recall']:.4f} "
        f"F1={val_metrics['f1']:.4f} F2={val_metrics['f2']:.4f} AUC={val_metrics['auc']:.4f}"
    )

    if val_metrics["f2"] > best_val_f2:
        best_val_f2 = val_metrics["f2"]
        torch.save(
            {
                "model_name": MODEL_NAME,
                "feature_dim": FEATURE_DIM,
                "hidden_dim": HIDDEN_DIM,
                "num_classes": HEAD_NUM_CLASSES,
                "class_names": CLASS_NAMES,
                "classifier_state_dict": model.head.state_dict(),
                "best_val_f2": best_val_f2,
                "epoch": epoch,
                "config": {
                    "patch_size": PATCH_SIZE,
                    "stride": STRIDE,
                    "model_input_size": MODEL_INPUT_SIZE,
                    "bands": BANDS,
                    "normalize_mode": NORMALIZE_MODE,
                    "pair_mode": PAIR_MODE,
                    "lead_days": LEAD_DAYS,
                    "fire_threshold": FIRE_THRESHOLD,
                    "fbeta_beta": FBETA_BETA,
                    "positive_weight_multiplier": POSITIVE_WEIGHT_MULTIPLIER,
                    "min_positive_pixels": MIN_POSITIVE_PIXELS,
                    "negative_keep_ratio": NEGATIVE_KEEP_RATIO,
                    "unfreeze_last_backbone_blocks": UNFREEZE_LAST_BACKBONE_BLOCKS,
                    "backbone_learning_rate": BACKBONE_LEARNING_RATE,
                    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
                },
            },
            BEST_HEAD_PATH,
        )
        print("Saved current best classifier head:", BEST_HEAD_PATH)

history_df = pd.DataFrame(history)
display(history_df.tail())


In [ ]:
# =========================
# 9. 训练曲线
# =========================

if "history_df" not in globals() or len(history_df) == 0:
    raise ValueError("history_df 为空，请先运行训练单元。")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="s", label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_acc"], marker="o", label="Train Acc")
axes[1].plot(history_df["epoch"], history_df["val_acc"], marker="s", label="Val Acc")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(history_df["epoch"], history_df["val_precision"], marker="o", label="Precision")
axes[2].plot(history_df["epoch"], history_df["val_recall"], marker="s", label="Recall")
axes[2].plot(history_df["epoch"], history_df["val_f1"], marker="^", label="F1")
axes[2].plot(history_df["epoch"], history_df["val_f2"], marker="d", label="F2")
axes[2].plot(history_df["epoch"], history_df["val_auc"], marker="x", label="AUC")
axes[2].set_title("Validation Metrics")
axes[2].set_xlabel("Epoch")
axes[2].set_ylim(0, 1.05)
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
# =========================
# 10. Test prediction analysis: metrics, confusion matrix, confidence
# =========================


def batch_value(batch, key, idx):
    value = batch[key]
    if isinstance(value, torch.Tensor):
        return value[idx].item()
    return value[idx]


def predict_loader(model, loader, desc="predict"):
    if loader is None:
        return pd.DataFrame()
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            images = batch["image"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast(AMP_DEVICE_TYPE, enabled=(AMP_DEVICE_TYPE == "cuda")):
                logits = model(images)
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            preds = threshold_predict_from_probs(probs)
            argmax_preds = probs.argmax(axis=1)
            confs = binary_confidence_from_probs(probs, preds)
            batch_size = images.size(0)
            for i in range(batch_size):
                row = {
                    "image_path": batch_value(batch, "image_path", i),
                    "label_path": batch_value(batch, "label_path", i),
                    "x": int(batch_value(batch, "x", i)),
                    "y": int(batch_value(batch, "y", i)),
                    "positive_pixels": int(batch_value(batch, "positive_pixels", i)),
                    "label": int(batch_value(batch, "label", i)),
                    "pred": int(preds[i]),
                    "pred_argmax": int(argmax_preds[i]),
                    "fire_threshold": float(FIRE_THRESHOLD),
                    "confidence": float(confs[i]),
                }
                for class_idx, class_name in enumerate(CLASS_NAMES[:probs.shape[1]]):
                    row[f"prob_{class_name}"] = float(probs[i, class_idx])
                rows.append(row)
    return pd.DataFrame(rows)


if BEST_HEAD_PATH.exists():
    load_classifier_head(model, BEST_HEAD_PATH, strict=True)
else:
    print("BEST_HEAD_PATH was not found. Using the model currently in memory.")

analysis_loader = test_loader if test_loader is not None else val_loader
analysis_name = "test" if test_loader is not None else "val"
pred_df = predict_loader(model, analysis_loader, desc=f"predict {analysis_name}")

if len(pred_df) == 0:
    raise ValueError("No prediction samples are available for analysis.")

pred_df["correct"] = pred_df["label"] == pred_df["pred"]
pred_df.to_csv(PREDICTION_CSV_PATH, index=False, encoding="utf-8-sig")
print("Prediction CSV saved:", PREDICTION_CSV_PATH.resolve())
print("Current FIRE_THRESHOLD:", FIRE_THRESHOLD)

y_true = pred_df["label"].to_numpy()
y_pred = pred_df["pred"].to_numpy()
prob_col = f"prob_{CLASS_NAMES[POSITIVE_CLASS_INDEX]}"
y_prob = pred_df[prob_col].to_numpy() if prob_col in pred_df.columns else None
metrics = compute_metrics(y_true, y_pred, y_prob=np.column_stack([1 - y_prob, y_prob]) if y_prob is not None else None)

print(classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES, zero_division=0))
print("Accuracy:", round(metrics["acc"], 4))
print("Precision:", round(metrics["precision"], 4))
print("Recall:", round(metrics["recall"], 4))
print("F1:", round(metrics["f1"], 4))
print("F2:", round(metrics["f2"], 4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title(f"{analysis_name} Confusion Matrix")
axes[0].set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=30, ha="right")
axes[0].set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        axes[0].text(col, row, cm[row, col], ha="center", va="center")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

pred_df["confidence"].hist(ax=axes[1], bins=20)
axes[1].set_title("Prediction Confidence")
axes[1].set_xlabel("confidence")
axes[1].set_ylabel("count")

if prob_col in pred_df.columns:
    axes[2].hist(pred_df.loc[pred_df["label"] == 0, prob_col], bins=20, alpha=0.6, label=CLASS_NAMES[0])
    axes[2].hist(pred_df.loc[pred_df["label"] == POSITIVE_CLASS_INDEX, prob_col], bins=20, alpha=0.6, label=CLASS_NAMES[POSITIVE_CLASS_INDEX])
    axes[2].axvline(FIRE_THRESHOLD, color="red", linestyle="--", label=f"threshold={FIRE_THRESHOLD}")
    axes[2].set_title("Positive-class Probability")
    axes[2].set_xlabel(prob_col)
    axes[2].legend()
else:
    axes[2].axis("off")

plt.tight_layout()
plt.show()
display(pred_df.head(10))
display(pred_df.sort_values("confidence").head(10))


In [ ]:
# =========================
# 11. 单幅影像滑窗预测：生成火灾概率图
# =========================


def read_display_rgb(image_path: Path):
    with rasterio.open(image_path) as src:
        indexes = [idx + 1 for idx in DISPLAY_BANDS_ZERO_BASED]
        indexes = [idx for idx in indexes if idx <= src.count]
        img = src.read(indexes=indexes).astype(np.float32)
    img = preprocess_patch(img)
    if img.shape[0] == 1:
        rgb = np.repeat(img, 3, axis=0)
    elif img.shape[0] == 2:
        rgb = np.concatenate([img, img[:1]], axis=0)
    else:
        rgb = img[:3]
    rgb = np.transpose(rgb, (1, 2, 0))
    rgb = np.clip(rgb * 2.5, 0, 1)
    return rgb


def infer_patch_batch(model, patch_tensors):
    batch_tensor = torch.stack(patch_tensors, dim=0).to(DEVICE)
    with torch.no_grad(), torch.amp.autocast(AMP_DEVICE_TYPE, enabled=(AMP_DEVICE_TYPE == "cuda")):
        logits = model(batch_tensor)
        probs = torch.softmax(logits, dim=1)[:, POSITIVE_CLASS_INDEX]
    return probs.detach().cpu().numpy()


def predict_scene_probability(model, image_path: Path, output_tif_path: Path | None = None):
    model.eval()

    with rasterio.open(image_path) as src:
        height, width = src.height, src.width
        profile = src.profile.copy()
        prob_sum = np.zeros((height, width), dtype=np.float32)
        prob_count = np.zeros((height, width), dtype=np.float32)

        patch_tensors = []
        coords = []

        def flush_batch():
            if not patch_tensors:
                return
            probs = infer_patch_batch(model, patch_tensors)
            for (x, y), prob in zip(coords, probs):
                prob_sum[y:y + PATCH_SIZE, x:x + PATCH_SIZE] += float(prob)
                prob_count[y:y + PATCH_SIZE, x:x + PATCH_SIZE] += 1.0
            patch_tensors.clear()
            coords.clear()

        ys = list(range(0, height - PATCH_SIZE + 1, STRIDE))
        xs = list(range(0, width - PATCH_SIZE + 1, STRIDE))
        for y in tqdm(ys, desc="scene rows"):
            for x in xs:
                img = src.read(indexes=BANDS, window=Window(x, y, PATCH_SIZE, PATCH_SIZE))
                img = preprocess_patch(img)
                image_tensor = resize_for_prithvi(torch.tensor(img, dtype=torch.float32))
                patch_tensors.append(image_tensor)
                coords.append((x, y))
                if len(patch_tensors) >= PRED_BATCH_SIZE:
                    flush_batch()
        flush_batch()

    prob_map = prob_sum / np.maximum(prob_count, 1.0)
    prob_map[prob_count == 0] = np.nan

    if output_tif_path is not None:
        profile.update(count=1, dtype="float32", nodata=np.nan)
        with rasterio.open(output_tif_path, "w", **profile) as dst:
            dst.write(prob_map.astype(np.float32), 1)
        print("概率 GeoTIFF 已保存:", output_tif_path.resolve())

    return prob_map


# 默认展示测试集第一幅影像；你也可以手动指定 scene_image_path。
scene_image_path = test_pairs[0][0] if len(test_pairs) else pairs[0][0]
scene_label_path = test_pairs[0][1] if len(test_pairs) else pairs[0][1]
print("scene_image_path:", scene_image_path)

prob_map = predict_scene_probability(model, scene_image_path, SCENE_PROB_TIF_PATH)
alert_map = prob_map >= FIRE_THRESHOLD
rgb = read_display_rgb(scene_image_path)

with rasterio.open(scene_label_path) as src:
    label_map = src.read(1)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(rgb)
axes[0].set_title("Image")
axes[0].axis("off")

axes[1].imshow(label_map, cmap="gray")
axes[1].set_title("Ground Truth")
axes[1].axis("off")

im = axes[2].imshow(prob_map, cmap="hot", vmin=0, vmax=1)
axes[2].set_title("Predicted Fire Probability")
axes[2].axis("off")
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

axes[3].imshow(alert_map, cmap="Reds", vmin=0, vmax=1)
axes[3].set_title(f"Fire Alert Map threshold={FIRE_THRESHOLD}")
axes[3].axis("off")

plt.tight_layout()
plt.show()


## 使用建议

- 如果只是用现有分类头做预测，把 `LOAD_EXISTING_HEAD=True`，可以跳过训练单元，直接运行测试集预测分析或整幅影像滑窗预测。
- 如果要正式训练，先运行到第 8 节，训练完成后会保存 `prithvi_patch_classifier_head_formal_best.pth`。
- `CLASS_NAMES` 只影响输出显示和 CSV 列名，不会自动改变模型输出顺序。当前默认是 0=非火灾，1=火灾。
- 如果正负样本极不平衡，优先调整 `NEGATIVE_KEEP_RATIO`、`MIN_POSITIVE_PIXELS` 和 `MAX_PATCHES_PER_IMAGE`。


In [ ]:
# 看 test 真实火灾样本的预测概率
fire_test = pred_df[pred_df["label"] == 1]
fire_test["prob_fire"].describe()

In [ ]:
plt.hist(fire_test["prob_fire"], bins=30)
plt.axvline(FIRE_THRESHOLD, color="red", linestyle="-")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score
import matplotlib.pyplot as plt

# 计算ROC曲线和AUC
# 处理y_prob可能是1维或2维的情况
y_proba = y_prob[:, 1] if y_prob.ndim > 1 else y_prob

fpr, tpr, thresholds = roc_curve(y_true, y_proba)
roc_auc = auc(fpr, tpr)
auc_score = roc_auc_score(y_true, y_proba)

print(f"AUC Score: {auc_score:.4f}")

# 绘制ROC曲线
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()